In [1]:
# Install deps (run once):
!pip install google-api-python-client youtube-transcript-api isodate pandas openpyxl

Defaulting to user installation because normal site-packages is not writeable


In [4]:
# ============================
# YouTube Channel Scanner + Transcript Saver (Jupyter-friendly)
# - Run log file: out_dir/run_YYYYMMDD_HHMMSS.log
# - Full status report: out_dir/full_status_report.csv (+ .xlsx)
# - Manifest: out_dir/manifest.csv
# - SAFE transcript providers:
#     1) CacheProvider
#     2) Official captions via OAuth (only if authorized)
#     3) Local ASR via Whisper (only from local audio files you provide)
# ============================

from __future__ import annotations

import json
import logging
import random
import re
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import isodate
import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# Optional OAuth imports
try:
    from google.oauth2.credentials import Credentials
    from google_auth_oauthlib.flow import InstalledAppFlow
    from google.auth.transport.requests import Request
except Exception:
    Credentials = None  # type: ignore

RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")


# -----------------------------
# Logging
# -----------------------------
LOG = logging.getLogger("yt_capture")
LOG.setLevel(logging.INFO)
if not LOG.handlers:
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    LOG.addHandler(ch)


def init_run_logger(out_dir: Path) -> Path:
    """Create file log handler and return its path."""
    out_dir.mkdir(parents=True, exist_ok=True)
    log_path = out_dir / f"run_{RUN_TS}.log"

    # avoid duplicates in notebooks
    for h in list(LOG.handlers):
        if isinstance(h, logging.FileHandler):
            return log_path

    fh = logging.FileHandler(log_path, encoding="utf-8")
    fh.setLevel(logging.INFO)
    fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    LOG.addHandler(fh)

    LOG.info("[LOG] Run log started -> %s", log_path)
    return log_path


# -----------------------------
# Config
# -----------------------------
@dataclass
class CaptureConfig:
    api_key: str
    registry_xlsx: str
    out_dir: str
    sheet: Optional[str] = None
    included_only: bool = True

    min_minutes: float = 5.0
    per_channel: int = 10
    scan_recent: int = 80

    keywords: Optional[List[str]] = None
    match_description: bool = True

    # resume / numbering
    start_index: int = 1
    state_filename: str = "_state.json"

    # YouTube API robustness
    max_retries: int = 8
    base_sleep: float = 1.0
    jitter: float = 0.4

    # Providers
    enable_cache_provider: bool = True

    # OAuth captions (ONLY if you have permission)
    enable_official_captions_provider: bool = False
    oauth_client_secrets_json: Optional[str] = None
    oauth_token_json: Optional[str] = None

    # Local ASR (ONLY from local audio you provide)
    enable_local_asr_provider: bool = False
    audio_dir: Optional[str] = None
    audio_extensions: Tuple[str, ...] = (".wav", ".mp3", ".m4a", ".flac", ".ogg")
    asr_language: Optional[str] = "en"
    whisper_model: str = "small"  # tiny/base/small/medium/large

    # Cache
    cache_dir: Optional[str] = None


# -----------------------------
# Utility helpers
# -----------------------------
def sanitize(s: str, max_len: int = 80) -> str:
    s = re.sub(r"[^\w\- ]+", "", s, flags=re.UNICODE).strip()
    s = re.sub(r"\s+", "_", s)
    return s[:max_len] if len(s) > max_len else s


def parse_channel_url(url: str) -> Tuple[Optional[str], Optional[str]]:
    if not isinstance(url, str):
        return None, None
    url = url.strip()
    m = re.search(r"(UC[a-zA-Z0-9_-]{20,})", url)
    uc = m.group(1) if m else None
    m = re.search(r"/@([a-zA-Z0-9_.-]+)", url)
    handle = m.group(1) if m else None
    return uc, handle


def chunked(seq: List[str], n: int) -> Iterable[List[str]]:
    for i in range(0, len(seq), n):
        yield seq[i : i + n]


def keyword_match(text: str, keywords: List[str]) -> bool:
    if not keywords:
        return True
    t = (text or "").lower()
    return any(k in t for k in keywords)


# -----------------------------
# Robust execute with retries
# -----------------------------
def execute_with_retries(request, *, cfg: CaptureConfig, context: str, ctx: Optional[Dict] = None):
    """
    ctx (optional) collects:
      - retry_count
      - last_http_status
      - last_error
    """
    if ctx is None:
        ctx = {}
    ctx.setdefault("retry_count", 0)
    ctx.setdefault("last_http_status", "")
    ctx.setdefault("last_error", "")

    for attempt in range(cfg.max_retries):
        try:
            return request.execute()
        except HttpError as e:
            status = getattr(e.resp, "status", None)
            ctx["last_http_status"] = status if status is not None else ""
            ctx["last_error"] = str(e)

            retryable = status in (403, 429, 500, 502, 503, 504)
            if (not retryable) or attempt == cfg.max_retries - 1:
                LOG.error("HTTP error in %s: status=%s, error=%s", context, status, str(e))
                raise

            ctx["retry_count"] += 1
            sleep = cfg.base_sleep * (2 ** attempt)
            sleep *= 1.0 + random.uniform(-cfg.jitter, cfg.jitter)
            sleep = min(sleep, 60.0)
            LOG.warning("Retrying %s after HttpError %s (attempt %d/%d), sleep %.1fs",
                        context, status, attempt + 1, cfg.max_retries, sleep)
            time.sleep(sleep)
        except Exception as e:
            ctx["last_error"] = repr(e)
            if attempt == cfg.max_retries - 1:
                LOG.error("Non-HTTP error in %s: %s", context, repr(e))
                raise

            ctx["retry_count"] += 1
            sleep = cfg.base_sleep * (2 ** attempt)
            sleep *= 1.0 + random.uniform(-cfg.jitter, cfg.jitter)
            sleep = min(sleep, 30.0)
            LOG.warning("Retrying %s after %s (attempt %d/%d), sleep %.1fs",
                        context, type(e).__name__, attempt + 1, cfg.max_retries, sleep)
            time.sleep(sleep)


# -----------------------------
# YouTube clients
# -----------------------------
SCOPES = ["https://www.googleapis.com/auth/youtube.force-ssl"]


def build_youtube_api_key_client(cfg: CaptureConfig):
    return build("youtube", "v3", developerKey=cfg.api_key)


def build_youtube_oauth_client(cfg: CaptureConfig):
    if Credentials is None:
        raise RuntimeError("OAuth requested but google-auth libraries not available.")
    if not cfg.oauth_client_secrets_json:
        raise ValueError("OAuth requires oauth_client_secrets_json path.")

    token_path = Path(cfg.oauth_token_json or "token.json")
    creds = None
    if token_path.exists():
        creds = Credentials.from_authorized_user_file(str(token_path), SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(cfg.oauth_client_secrets_json, SCOPES)
            creds = flow.run_local_server(port=0)
        token_path.write_text(creds.to_json(), encoding="utf-8")

    return build("youtube", "v3", credentials=creds)


# -----------------------------
# Channel/video listing
# -----------------------------
def resolve_channel_uc(youtube, channel_url: str, channel_name: str, cfg: CaptureConfig, req_ctx: Dict) -> Optional[str]:
    uc, handle = parse_channel_url(channel_url)
    if uc:
        return uc

    if handle:
        try:
            req = youtube.channels().list(part="id", forHandle=handle)
            resp = execute_with_retries(req, cfg=cfg, context=f"channels.list(forHandle={handle})", ctx=req_ctx)
            items = resp.get("items", [])
            if items:
                return items[0]["id"]
        except Exception:
            pass

    # ambiguous fallback
    try:
        req = youtube.search().list(part="id", q=channel_name, type="channel", maxResults=1)
        resp = execute_with_retries(req, cfg=cfg, context=f"search.list(channel q={channel_name})", ctx=req_ctx)
        items = resp.get("items", [])
        if items:
            return items[0]["id"]["channelId"]
    except Exception:
        pass

    return None


def get_uploads_playlist_id(youtube, channel_uc: str, cfg: CaptureConfig, req_ctx: Dict) -> Optional[str]:
    req = youtube.channels().list(part="contentDetails", id=channel_uc)
    resp = execute_with_retries(req, cfg=cfg, context=f"channels.list(contentDetails id={channel_uc})", ctx=req_ctx)
    items = resp.get("items", [])
    if not items:
        return None
    return items[0]["contentDetails"]["relatedPlaylists"]["uploads"]


def list_video_ids_from_uploads(youtube, uploads_playlist_id: str, max_videos: int, cfg: CaptureConfig, req_ctx: Dict) -> List[str]:
    video_ids: List[str] = []
    page_token = None
    while len(video_ids) < max_videos:
        req = youtube.playlistItems().list(
            part="contentDetails",
            playlistId=uploads_playlist_id,
            maxResults=min(50, max_videos - len(video_ids)),
            pageToken=page_token,
        )
        resp = execute_with_retries(req, cfg=cfg, context=f"playlistItems.list({uploads_playlist_id})", ctx=req_ctx)
        for item in resp.get("items", []):
            vid = (item.get("contentDetails") or {}).get("videoId")
            if vid:
                video_ids.append(vid)

        page_token = resp.get("nextPageToken")
        if not page_token:
            break
    return video_ids


def get_video_metadata(youtube, video_ids: List[str], cfg: CaptureConfig, req_ctx: Dict) -> List[Dict]:
    rows: List[Dict] = []
    for batch in chunked(video_ids, 50):
        req = youtube.videos().list(part="snippet,contentDetails,status", id=",".join(batch))
        resp = execute_with_retries(req, cfg=cfg, context="videos.list(snippet,contentDetails,status)", ctx=req_ctx)
        for v in resp.get("items", []):
            vid = v.get("id")
            snippet = v.get("snippet") or {}
            cd = v.get("contentDetails") or {}
            status = v.get("status") or {}
            if not vid or not snippet or not cd:
                continue

            dur_iso = cd.get("duration")
            if not dur_iso:
                continue
            try:
                dur_seconds = int(isodate.parse_duration(dur_iso).total_seconds())
            except Exception:
                continue

            published_at = snippet.get("publishedAt", "")
            year = int(published_at[:4]) if published_at else None

            rows.append({
                "video_id": vid,
                "title": snippet.get("title", ""),
                "description": snippet.get("description", ""),
                "published_at": published_at,
                "year": year,
                "duration_iso": dur_iso,
                "duration_seconds": dur_seconds,
                "privacy_status": status.get("privacyStatus"),
                "upload_status": status.get("uploadStatus"),
                # This flag is available via Data API (helps identify likely transcript availability in UI)
                "caption_flag": cd.get("caption"),  # "true"/"false" (string)
            })

    rows.sort(key=lambda r: r.get("published_at", ""), reverse=True)
    return rows


# -----------------------------
# State / numbering
# -----------------------------
def load_state(state_path: Path) -> Dict:
    if state_path.exists():
        try:
            return json.loads(state_path.read_text(encoding="utf-8"))
        except Exception:
            return {}
    return {}


def save_state(state_path: Path, state: Dict) -> None:
    tmp = state_path.with_suffix(".tmp")
    tmp.write_text(json.dumps(state, indent=2), encoding="utf-8")
    tmp.replace(state_path)


def next_global_counter(out_path: Path, start_index: int) -> int:
    nums: List[int] = []
    for p in out_path.rglob("V*.txt"):
        stem = p.stem
        if stem.startswith("V") and stem[1:].isdigit():
            nums.append(int(stem[1:]))
    return (max(nums) + 1) if nums else start_index


# -----------------------------
# Transcript providers (SAFE)
# -----------------------------
@dataclass
class TranscriptResult:
    ok: bool
    text: Optional[str]
    provider: str
    reason: str


class TranscriptProvider:
    name: str
    def fetch(self, *, youtube_id: str, dest_txt: Path, cfg: CaptureConfig, ctx: Dict) -> TranscriptResult:
        raise NotImplementedError


class CacheProvider(TranscriptProvider):
    name = "cache"

    def fetch(self, *, youtube_id: str, dest_txt: Path, cfg: CaptureConfig, ctx: Dict) -> TranscriptResult:
        if dest_txt.exists():
            try:
                t = dest_txt.read_text(encoding="utf-8", errors="ignore").strip()
                if t and "NO TRANSCRIPT AVAILABLE" not in t and "TRANSCRIPT NOT FETCHED" not in t:
                    return TranscriptResult(True, t, self.name, "used_existing_dest")
            except Exception:
                pass

        cache_root = Path(cfg.cache_dir) if cfg.cache_dir else Path(cfg.out_dir)
        alt = cache_root / f"{youtube_id}.txt"
        if alt.exists():
            try:
                t = alt.read_text(encoding="utf-8", errors="ignore")
                if t.strip():
                    return TranscriptResult(True, t, self.name, "found_in_cache_root")
                return TranscriptResult(False, None, self.name, "cache_empty")
            except Exception as e:
                return TranscriptResult(False, None, self.name, f"cache_read_error:{type(e).__name__}")

        return TranscriptResult(False, None, self.name, "miss")


class OfficialYouTubeCaptionsProvider(TranscriptProvider):
    name = "official_captions_oauth"

    def __init__(self, youtube_oauth):
        self.youtube_oauth = youtube_oauth

    def fetch(self, *, youtube_id: str, dest_txt: Path, cfg: CaptureConfig, ctx: Dict) -> TranscriptResult:
        try:
            req_ctx = ctx.setdefault("yt_req_ctx", {})
            req = self.youtube_oauth.captions().list(part="id,snippet", videoId=youtube_id)
            resp = execute_with_retries(req, cfg=cfg, context=f"captions.list(videoId={youtube_id})", ctx=req_ctx)
            items = resp.get("items", [])
            if not items:
                return TranscriptResult(False, None, self.name, "no_caption_tracks")

            def score(it):
                sn = it.get("snippet", {}) or {}
                lang = (sn.get("language") or "").lower()
                name = (sn.get("name") or "").lower()
                return (0 if lang.startswith("en") else 1, 0 if "auto" not in name else 1)

            items.sort(key=score)
            caption_id = items[0]["id"]

            req = self.youtube_oauth.captions().download(id=caption_id, tfmt="vtt")
            data = execute_with_retries(req, cfg=cfg, context=f"captions.download(id={caption_id})", ctx=req_ctx)

            text = data.decode("utf-8", errors="ignore") if isinstance(data, bytes) else str(data)
            if not text.strip():
                return TranscriptResult(False, None, self.name, "download_empty")

            return TranscriptResult(True, text, self.name, "downloaded_vtt")
        except HttpError as e:
            status = getattr(e.resp, "status", None)
            return TranscriptResult(False, None, self.name, f"http_{status}")
        except Exception as e:
            return TranscriptResult(False, None, self.name, f"error_{type(e).__name__}")


class LocalASRProvider(TranscriptProvider):
    name = "local_asr_whisper"

    def _find_audio(self, youtube_id: str, cfg: CaptureConfig) -> Optional[Path]:
        if not cfg.audio_dir:
            return None
        root = Path(cfg.audio_dir)
        for ext in cfg.audio_extensions:
            p = root / f"{youtube_id}{ext}"
            if p.exists():
                return p
        return None

    def _asr_transcribe(self, audio_path: Path, cfg: CaptureConfig) -> str:
        """
        Uses Whisper to transcribe LOCAL audio into timestamped text like YouTube's transcript UI.
        Requires:
          pip install -U openai-whisper
          ffmpeg installed on system
        """
        import whisper

        model = whisper.load_model(cfg.whisper_model)
        result = model.transcribe(
            str(audio_path),
            language=cfg.asr_language,
            fp16=False,
            verbose=False,
        )

        lines: List[str] = []
        for seg in result.get("segments", []) or []:
            start = float(seg.get("start", 0.0))
            text = (seg.get("text") or "").strip()
            if not text:
                continue
            mm = int(start // 60)
            ss = int(start % 60)
            ts = f"{mm}:{ss:02d}"
            lines.append(ts)
            lines.append(text)

        return ("\n".join(lines).strip() + "\n") if lines else ""

    def fetch(self, *, youtube_id: str, dest_txt: Path, cfg: CaptureConfig, ctx: Dict) -> TranscriptResult:
        audio = self._find_audio(youtube_id, cfg)
        if not audio:
            return TranscriptResult(False, None, self.name, "no_local_audio")

        try:
            text = self._asr_transcribe(audio, cfg)
            if not text.strip():
                return TranscriptResult(False, None, self.name, "empty_asr_result")
            return TranscriptResult(True, text, self.name, f"asr_ok:{audio.name}")
        except Exception as e:
            return TranscriptResult(False, None, self.name, f"asr_error:{type(e).__name__}")


def build_providers(cfg: CaptureConfig, youtube_oauth=None) -> List[TranscriptProvider]:
    providers: List[TranscriptProvider] = []
    if cfg.enable_cache_provider:
        providers.append(CacheProvider())
    if cfg.enable_official_captions_provider and youtube_oauth is not None:
        providers.append(OfficialYouTubeCaptionsProvider(youtube_oauth))
    if cfg.enable_local_asr_provider:
        providers.append(LocalASRProvider())
    return providers


def fetch_transcript_via_chain(
    *,
    youtube_id: str,
    dest_txt: Path,
    cfg: CaptureConfig,
    providers: List[TranscriptProvider],
    ctx: Dict
) -> TranscriptResult:
    last = TranscriptResult(False, None, "none", "no_providers")
    for p in providers:
        res = p.fetch(youtube_id=youtube_id, dest_txt=dest_txt, cfg=cfg, ctx=ctx)
        if res.ok and res.text:
            return res
        last = res
    return last


# -----------------------------
# Main runner
# -----------------------------
def run_transcript_capture(cfg: CaptureConfig) -> pd.DataFrame:
    out_path = Path(cfg.out_dir)
    log_file_path = init_run_logger(out_path)

    keywords = cfg.keywords or [
        "ransomware", "extortion", "double extortion", "locker",
        "encrypt", "encryption", "decrypt", "data leak",
        "incident response", "ir",
        "lockbit", "alphv", "blackcat", "conti", "ryuk",
        "cl0p", "clop", "akira", "revil", "sodinokibi",
    ]
    keywords = [k.strip().lower() for k in keywords if k.strip()]
    min_seconds = int(cfg.min_minutes * 60)
    scan_n = max(cfg.scan_recent, cfg.per_channel)

    state_path = out_path / cfg.state_filename
    state = load_state(state_path)

    global_counter = state.get("global_counter")
    if not isinstance(global_counter, int):
        global_counter = next_global_counter(out_path, cfg.start_index)

    youtube = build_youtube_api_key_client(cfg)

    youtube_oauth = None
    if cfg.enable_official_captions_provider:
        youtube_oauth = build_youtube_oauth_client(cfg)

    providers = build_providers(cfg, youtube_oauth=youtube_oauth)

    df = pd.read_excel(cfg.registry_xlsx, sheet_name=cfg.sheet)
    required_cols = {"Channel_ID", "Channel_Name", "Channel_URL"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Registry missing required columns: {sorted(missing)}")

    if cfg.included_only and "Included (Yes/No)" in df.columns:
        df = df[df["Included (Yes/No)"].astype(str).str.strip().str.lower().eq("yes")].copy()

    manifest_rows: List[Dict] = []
    status_rows: List[Dict] = []

    for _, row in df.iterrows():
        channel_id = str(row["Channel_ID"]).strip()
        channel_name = str(row["Channel_Name"]).strip()
        channel_url = str(row["Channel_URL"]).strip()

        LOG.info("Channel %s | %s", channel_id, channel_name)

        yt_req_ctx: Dict = {"retry_count": 0, "last_http_status": "", "last_error": ""}

        # Resolve channel UC
        try:
            channel_uc = resolve_channel_uc(youtube, channel_url, channel_name, cfg, yt_req_ctx)
        except Exception as e:
            LOG.error("[FAIL] resolve_channel_uc %s | %s: %s", channel_id, channel_name, repr(e))
            channel_uc = None

        if not channel_uc:
            status_rows.append({
                "Run_TS": RUN_TS, "Channel_ID": channel_id, "Channel_Name": channel_name, "Channel_URL": channel_url,
                "Channel_UC": "", "YouTube_Video_ID": "", "Stage": "resolve_channel", "Outcome": "fail",
                "Failure_Reason": "could_not_resolve_channel_uc",
                "HTTP_Status": yt_req_ctx.get("last_http_status", ""),
                "Retry_Count": yt_req_ctx.get("retry_count", 0),
                "Provider": "", "Transcript_Available": False, "Output_File": "",
            })
            continue

        # Uploads playlist
        try:
            uploads = get_uploads_playlist_id(youtube, channel_uc, cfg, yt_req_ctx)
        except Exception as e:
            LOG.error("[FAIL] get_uploads_playlist %s (%s): %s", channel_id, channel_uc, repr(e))
            uploads = None

        if not uploads:
            status_rows.append({
                "Run_TS": RUN_TS, "Channel_ID": channel_id, "Channel_Name": channel_name, "Channel_URL": channel_url,
                "Channel_UC": channel_uc, "YouTube_Video_ID": "", "Stage": "get_uploads_playlist", "Outcome": "fail",
                "Failure_Reason": "could_not_get_uploads_playlist",
                "HTTP_Status": yt_req_ctx.get("last_http_status", ""),
                "Retry_Count": yt_req_ctx.get("retry_count", 0),
                "Provider": "", "Transcript_Available": False, "Output_File": "",
            })
            continue

        # Video IDs
        try:
            video_ids = list_video_ids_from_uploads(youtube, uploads, max_videos=scan_n, cfg=cfg, req_ctx=yt_req_ctx)
        except Exception as e:
            LOG.error("[FAIL] list_video_ids_from_uploads %s (%s): %s", channel_id, uploads, repr(e))
            video_ids = []

        if not video_ids:
            status_rows.append({
                "Run_TS": RUN_TS, "Channel_ID": channel_id, "Channel_Name": channel_name, "Channel_URL": channel_url,
                "Channel_UC": channel_uc, "YouTube_Video_ID": "", "Stage": "list_videos", "Outcome": "fail",
                "Failure_Reason": "no_videos_returned",
                "HTTP_Status": yt_req_ctx.get("last_http_status", ""),
                "Retry_Count": yt_req_ctx.get("retry_count", 0),
                "Provider": "", "Transcript_Available": False, "Output_File": "",
            })
            continue

        # Metadata
        try:
            meta = get_video_metadata(youtube, video_ids, cfg, yt_req_ctx)
        except Exception as e:
            LOG.error("[FAIL] get_video_metadata %s: %s", channel_id, repr(e))
            meta = []

        if not meta:
            status_rows.append({
                "Run_TS": RUN_TS, "Channel_ID": channel_id, "Channel_Name": channel_name, "Channel_URL": channel_url,
                "Channel_UC": channel_uc, "YouTube_Video_ID": "", "Stage": "video_metadata", "Outcome": "fail",
                "Failure_Reason": "no_metadata_rows",
                "HTTP_Status": yt_req_ctx.get("last_http_status", ""),
                "Retry_Count": yt_req_ctx.get("retry_count", 0),
                "Provider": "", "Transcript_Available": False, "Output_File": "",
            })
            continue

        # Select videos (and log skips for ALL scanned videos)
        selected: List[Dict] = []
        for m in meta:
            yt_vid = m["video_id"]
            title = m.get("title", "")
            desc = m.get("description", "")
            dur_s = int(m.get("duration_seconds", 0) or 0)

            if dur_s < min_seconds:
                status_rows.append({
                    "Run_TS": RUN_TS, "Channel_ID": channel_id, "Channel_Name": channel_name,
                    "Channel_URL": channel_url, "Channel_UC": channel_uc, "YouTube_Video_ID": yt_vid,
                    "Video_Title": title, "PublishedAt": m.get("published_at", ""), "DurationSeconds": dur_s,
                    "KeywordMatch": False, "Selected": False, "Stage": "filter", "Outcome": "skip",
                    "Failure_Reason": f"duration_below_min:{min_seconds}",
                    "HTTP_Status": "", "Retry_Count": yt_req_ctx.get("retry_count", 0),
                    "Provider": "", "Transcript_Available": False, "Output_File": "",
                })
                continue

            text = title + ("\n" + desc if cfg.match_description else "")
            kw_match = keyword_match(text, keywords)
            if not kw_match:
                status_rows.append({
                    "Run_TS": RUN_TS, "Channel_ID": channel_id, "Channel_Name": channel_name,
                    "Channel_URL": channel_url, "Channel_UC": channel_uc, "YouTube_Video_ID": yt_vid,
                    "Video_Title": title, "PublishedAt": m.get("published_at", ""), "DurationSeconds": dur_s,
                    "KeywordMatch": False, "Selected": False, "Stage": "filter", "Outcome": "skip",
                    "Failure_Reason": "keyword_no_match",
                    "HTTP_Status": "", "Retry_Count": yt_req_ctx.get("retry_count", 0),
                    "Provider": "", "Transcript_Available": False, "Output_File": "",
                })
                continue

            t = text.lower()
            matched = sorted({k for k in keywords if k in t})
            m["matched_keywords"] = ";".join(matched)
            selected.append(m)

        selected = selected[: cfg.per_channel]

        channel_folder = out_path / f"{channel_id}_{sanitize(channel_name)}"
        channel_folder.mkdir(parents=True, exist_ok=True)

        kept = 0
        for m in selected:
            yt_vid = m["video_id"]
            title = m.get("title", "")
            year = m.get("year")
            published_at = m.get("published_at", "")
            dur_s = m.get("duration_seconds", 0)
            dur_iso = m.get("duration_iso", "")
            matched_kw = m.get("matched_keywords", "")
            caption_flag = m.get("caption_flag", "")

            vid_id = f"V{global_counter:04d}"
            txt_file = channel_folder / f"{vid_id}.txt"
            meta_file = channel_folder / f"{vid_id}.meta.json"

            if txt_file.exists() or meta_file.exists():
                LOG.info("[SKIP] %s exists in %s", vid_id, channel_folder)
                global_counter += 1
                continue

            ctx = {"yt_req_ctx": {"retry_count": 0, "last_http_status": "", "last_error": ""}}

            failure_stage = ""
            http_status = ""
            provider = "none"
            reason = ""
            transcript_text: Optional[str] = None

            try:
                res = fetch_transcript_via_chain(
                    youtube_id=yt_vid, dest_txt=txt_file, cfg=cfg, providers=providers, ctx=ctx
                )
                provider, reason, transcript_text = res.provider, res.reason, res.text
            except HttpError as e:
                failure_stage = "transcript"
                http_status = getattr(e.resp, "status", "") or ""
                provider = "exception"
                reason = f"HttpError_{http_status}"
                transcript_text = None
            except Exception as e:
                failure_stage = "transcript"
                provider = "exception"
                reason = f"{type(e).__name__}:{str(e)}"
                transcript_text = None

            transcript_available = bool(transcript_text and transcript_text.strip())

            if not transcript_available:
                transcript_text = (
                    "[NO TRANSCRIPT AVAILABLE]\n"
                    f"Provider: {provider}\n"
                    f"Reason: {reason}\n"
                    f"CaptionFlag(contentDetails.caption): {caption_flag}\n"
                    "Notes:\n"
                    "- API key-only access cannot download transcripts for channels you do not control.\n"
                    "- Official captions download requires OAuth authorization for that video/channel.\n"
                    "- Local ASR requires local audio files you provide (named <video_id>.<ext>).\n"
                )

            txt_file.write_text(transcript_text, encoding="utf-8")

            meta_payload = {
                "Video_ID": vid_id,
                "Channel_ID": channel_id,
                "Channel_Name": channel_name,
                "Channel_UC": channel_uc,
                "YouTube_Video_ID": yt_vid,
                "Video_Title": title,
                "Year": year,
                "PublishedAt": published_at,
                "DurationSeconds": dur_s,
                "DurationISO": dur_iso,
                "Matched_Keywords": matched_kw,
                "CaptionFlag": caption_flag,
                "Transcript_Available": transcript_available,
                "Transcript_Provider": provider,
                "Transcript_Reason": reason,
            }
            meta_file.write_text(json.dumps(meta_payload, indent=2), encoding="utf-8")

            manifest_rows.append({**meta_payload, "Transcript_Path": str(txt_file), "Meta_Path": str(meta_file)})

            req_ctx2 = ctx.get("yt_req_ctx", {}) or {}
            status_rows.append({
                "Run_TS": RUN_TS,
                "Channel_ID": channel_id,
                "Channel_Name": channel_name,
                "Channel_URL": channel_url,
                "Channel_UC": channel_uc,
                "YouTube_Video_ID": yt_vid,
                "Video_Title": title,
                "PublishedAt": published_at,
                "DurationSeconds": dur_s,
                "KeywordMatch": True,
                "Selected": True,
                "Stage": "transcript",
                "Outcome": "ok" if transcript_available else "fail",
                "Failure_Stage": failure_stage,
                "Failure_Reason": "" if transcript_available else f"{provider}::{reason}",
                "HTTP_Status": http_status or req_ctx2.get("last_http_status", ""),
                "Retry_Count": req_ctx2.get("retry_count", 0),
                "Provider": provider,
                "Transcript_Available": transcript_available,
                "Output_File": str(txt_file),
            })

            kept += 1
            global_counter += 1

            state["global_counter"] = global_counter
            save_state(state_path, state)

        LOG.info("[OK] %s | saved %d item(s) -> %s", channel_id, kept, channel_folder)

    manifest_df = pd.DataFrame(manifest_rows)
    status_df = pd.DataFrame(status_rows)

    manifest_csv = out_path / "manifest.csv"
    status_csv = out_path / "full_status_report.csv"
    status_xlsx = out_path / "full_status_report.xlsx"

    manifest_df.to_csv(manifest_csv, index=False, encoding="utf-8")
    status_df.to_csv(status_csv, index=False, encoding="utf-8")
    try:
        status_df.to_excel(status_xlsx, index=False)
    except Exception as e:
        LOG.warning("Could not write XLSX status report: %s", repr(e))

    LOG.info("[REPORT] Manifest CSV -> %s", manifest_csv)
    LOG.info("[REPORT] Status CSV   -> %s", status_csv)
    LOG.info("[REPORT] Status XLSX  -> %s", status_xlsx)
    LOG.info("[LOGFILE] Run log     -> %s", log_file_path)

    return manifest_df

In [5]:
cfg = CaptureConfig(
    api_key="YOUR_YOUTUBE_API_KEY",
    registry_xlsx="/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/rubrics/Dataset_Channel_Registry_Updated_50_fixed_urls.xlsx",
    sheet="Channel_Registry",
    out_dir="/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts",

    start_index=1,
    included_only=True,
    min_minutes=5,
    per_channel=10,
    scan_recent=80,
    match_description=True,

    enable_cache_provider=True,
    enable_official_captions_provider=False,

    enable_local_asr_provider=True,
    audio_dir="/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/audio_files",  # <-- your local audio folder
    whisper_model="small",
    asr_language="en",
)

manifest = run_transcript_capture(cfg)
manifest.head()


2026-02-25 15:08:06,745 | INFO | Channel C01 | The DFIR Report
2026-02-25 15:08:07,252 | INFO | [OK] C01 | saved 10 item(s) -> /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts/C01_The_DFIR_Report
2026-02-25 15:08:07,253 | INFO | Channel C02 | SANS Digital Forensics and Incident Response
2026-02-25 15:08:08,079 | INFO | [OK] C02 | saved 10 item(s) -> /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts/C02_SANS_Digital_Forensics_and_Incident_Response
2026-02-25 15:08:08,081 | INFO | Channel C03 | Black Hat
2026-02-25 15:08:08,735 | INFO | [OK] C03 | saved 10 item(s) -> /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts/C03_Black_Hat
2026-02-25 15:08:08,736 | INFO | Channel C04 | DEF CON Conference
2026-02-25 15:08:09,520 | INFO | [OK] C04 | saved 10 item(s) -> /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts/C04_DEF_CON_Conference
2026-02-25 15:08:09,

,Video_ID,Channel_ID,Channel_Name,Channel_UC,YouTube_Video_ID,Video_Title,Year,PublishedAt,DurationSeconds,DurationISO,Matched_Keywords,CaptionFlag,Transcript_Available,Transcript_Provider,Transcript_Reason,Transcript_Path,Meta_Path
0,V0001,C01,The DFIR Report,UC6R2MPMkkCqFxvAdQAI_23A,OfwHWjZvvuE,Blurring the Lines: Intrusion Shows Connection...,2025,2025-09-08T14:38:04Z,382,PT6M22S,ir;ransomware,false,False,local_asr_whisper,no_local_audio,/home/henrykabuye/Ransap/Publications 2026/Pro...,/home/henrykabuye/Ransap/Publications 2026/Pro...
1,V0002,C01,The DFIR Report,UC6R2MPMkkCqFxvAdQAI_23A,5fu-wYbBvuY,Hide Your RDP: Password Spray Leads to RansomH...,2025,2025-06-30T00:52:31Z,343,PT5M43S,ir;ransomware,false,False,local_asr_whisper,no_local_audio,/home/henrykabuye/Ransap/Publications 2026/Pro...,/home/henrykabuye/Ransap/Publications 2026/Pro...
2,V0003,C01,The DFIR Report,UC6R2MPMkkCqFxvAdQAI_23A,7HD-RYqIm3Q,Another Confluence Bites the Dust: Falling to ...,2025,2025-05-19T00:46:35Z,365,PT6M5S,ir;ransomware,false,False,local_asr_whisper,no_local_audio,/home/henrykabuye/Ransap/Publications 2026/Pro...,/home/henrykabuye/Ransap/Publications 2026/Pro...
3,V0004,C01,The DFIR Report,UC6R2MPMkkCqFxvAdQAI_23A,hpgjomPwHp4,Fake Zoom Ends in BlackSuit Ransomware,2025,2025-03-31T00:30:19Z,371,PT6M11S,ir;ransomware,false,False,local_asr_whisper,no_local_audio,/home/henrykabuye/Ransap/Publications 2026/Pro...,/home/henrykabuye/Ransap/Publications 2026/Pro...
4,V0005,C01,The DFIR Report,UC6R2MPMkkCqFxvAdQAI_23A,BMTtJLEJRGQ,Confluence Exploit Leads to LockBit Ransomware,2025,2025-02-24T00:33:55Z,341,PT5M41S,ir;lockbit;ransomware,false,False,local_asr_whisper,no_local_audio,/home/henrykabuye/Ransap/Publications 2026/Pro...,/home/henrykabuye/Ransap/Publications 2026/Pro...


In [6]:
from pathlib import Path
import pandas as pd

def build_manual_transcript_list(out_dir: str, out_name: str = "needs_manual_transcript_capture"):
    out_path = Path(out_dir)

    status_csv = out_path / "full_status_report.csv"
    if not status_csv.exists():
        raise FileNotFoundError(f"Could not find {status_csv}. Run run_transcript_capture(cfg) first.")

    df = pd.read_csv(status_csv)

    # Keep only videos selected but with no transcript available
    if "Selected" in df.columns:
        df = df[df["Selected"] == True]

    if "Transcript_Available" in df.columns:
        df = df[df["Transcript_Available"] == False]

    df["Video_URL"] = df["YouTube_Video_ID"].apply(
        lambda v: f"https://www.youtube.com/watch?v={v}" if isinstance(v, str) and v else ""
    )

    df["Manual_Action"] = "Open video → ⋯ → Show transcript → copy/paste into .txt"
    df["Suggested_FileName"] = df["YouTube_Video_ID"].apply(
        lambda v: f"{v}.txt" if isinstance(v, str) and v else ""
    )

    out_df = df[[
        "Channel_ID", "Channel_Name", "YouTube_Video_ID", "Video_URL",
        "Video_Title", "PublishedAt", "DurationSeconds",
        "Provider", "Failure_Reason",
        "Suggested_FileName", "Manual_Action"
    ]].drop_duplicates(subset=["YouTube_Video_ID"], keep="first")

    csv_path = out_path / f"{out_name}.csv"
    xlsx_path = out_path / f"{out_name}.xlsx"

    out_df.to_csv(csv_path, index=False, encoding="utf-8")
    out_df.to_excel(xlsx_path, index=False)

    return out_df, csv_path, xlsx_path


manual_df, csv_path, xlsx_path = build_manual_transcript_list(
    out_dir="/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts"
)

print("Saved CSV:", csv_path)
print("Saved XLSX:", xlsx_path)

manual_df.head(20)

Saved CSV: /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts/needs_manual_transcript_capture.csv
Saved XLSX: /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts/needs_manual_transcript_capture.xlsx


,Channel_ID,Channel_Name,YouTube_Video_ID,Video_URL,Video_Title,PublishedAt,DurationSeconds,Provider,Failure_Reason,Suggested_FileName,Manual_Action
7,C01,The DFIR Report,OfwHWjZvvuE,https://www.youtube.com/watch?v=OfwHWjZvvuE,Blurring the Lines: Intrusion Shows Connection...,2025-09-08T14:38:04Z,382,local_asr_whisper,local_asr_whisper::no_local_audio,OfwHWjZvvuE.txt,Open video → ⋯ → Show transcript → copy/paste ...
8,C01,The DFIR Report,5fu-wYbBvuY,https://www.youtube.com/watch?v=5fu-wYbBvuY,Hide Your RDP: Password Spray Leads to RansomH...,2025-06-30T00:52:31Z,343,local_asr_whisper,local_asr_whisper::no_local_audio,5fu-wYbBvuY.txt,Open video → ⋯ → Show transcript → copy/paste ...
9,C01,The DFIR Report,7HD-RYqIm3Q,https://www.youtube.com/watch?v=7HD-RYqIm3Q,Another Confluence Bites the Dust: Falling to ...,2025-05-19T00:46:35Z,365,local_asr_whisper,local_asr_whisper::no_local_audio,7HD-RYqIm3Q.txt,Open video → ⋯ → Show transcript → copy/paste ...
10,C01,The DFIR Report,hpgjomPwHp4,https://www.youtube.com/watch?v=hpgjomPwHp4,Fake Zoom Ends in BlackSuit Ransomware,2025-03-31T00:30:19Z,371,local_asr_whisper,local_asr_whisper::no_local_audio,hpgjomPwHp4.txt,Open video → ⋯ → Show transcript → copy/paste ...
11,C01,The DFIR Report,BMTtJLEJRGQ,https://www.youtube.com/watch?v=BMTtJLEJRGQ,Confluence Exploit Leads to LockBit Ransomware,2025-02-24T00:33:55Z,341,local_asr_whisper,local_asr_whisper::no_local_audio,BMTtJLEJRGQ.txt,Open video → ⋯ → Show transcript → copy/paste ...
12,C01,The DFIR Report,Fh_OcBYjzL8,https://www.youtube.com/watch?v=Fh_OcBYjzL8,DFIR Discussions: The Curious Case of an Egg-C...,2025-01-20T00:52:28Z,2392,local_asr_whisper,local_asr_whisper::no_local_audio,Fh_OcBYjzL8.txt,Open video → ⋯ → Show transcript → copy/paste ...
13,C01,The DFIR Report,CQ0zOWnHfHE,https://www.youtube.com/watch?v=CQ0zOWnHfHE,The Curious Case of an Egg-Cellent Resume,2024-12-02T02:14:13Z,431,local_asr_whisper,local_asr_whisper::no_local_audio,CQ0zOWnHfHE.txt,Open video → ⋯ → Show transcript → copy/paste ...
14,C01,The DFIR Report,7AhSj5GGVRw,https://www.youtube.com/watch?v=7AhSj5GGVRw,Nitrogen Campaign Drops Sliver and Ends With B...,2024-09-30T01:00:24Z,406,local_asr_whisper,local_asr_whisper::no_local_audio,7AhSj5GGVRw.txt,Open video → ⋯ → Show transcript → copy/paste ...
15,C01,The DFIR Report,t2sxRDE0Pr4,https://www.youtube.com/watch?v=t2sxRDE0Pr4,BlackSuit Ransomware,2024-08-26T00:41:51Z,316,local_asr_whisper,local_asr_whisper::no_local_audio,t2sxRDE0Pr4.txt,Open video → ⋯ → Show transcript → copy/paste ...
16,C01,The DFIR Report,wyg-YtiUFUo,https://www.youtube.com/watch?v=wyg-YtiUFUo,"Threat Actors' Toolkit: Leveraging Sliver, Pos...",2024-08-12T02:12:35Z,346,local_asr_whisper,local_asr_whisper::no_local_audio,wyg-YtiUFUo.txt,Open video → ⋯ → Show transcript → copy/paste ...


In [3]:
from pathlib import Path
import json
import pandas as pd

def build_paste_todo(out_dir: str, placeholder="NO TRANSCRIPT AVAILABLE"):
    out_path = Path(out_dir)
    meta_files = list(out_path.rglob("V*.meta.json"))

    rows = []
    for mf in meta_files:
        try:
            data = json.loads(mf.read_text(encoding="utf-8"))
            dfir_id = data.get("Video_ID", "")
            yt_id = data.get("YouTube_Video_ID", "")
            title = data.get("Video_Title", "")
            ch_name = data.get("Channel_Name", "")
            ch_id = data.get("Channel_ID", "")

            if not dfir_id or not yt_id:
                continue

            txt_path = mf.parent / f"{dfir_id}.txt"
            needs_paste = True
            current_preview = ""

            if txt_path.exists():
                t = txt_path.read_text(encoding="utf-8", errors="ignore")
                current_preview = t[:200].replace("\n", "\\n")
                # If placeholder exists, we still need to paste
                needs_paste = placeholder in t

            rows.append({
                "Channel_ID": ch_id,
                "Channel_Name": ch_name,
                "DFIR_ID": dfir_id,
                "YouTube_Video_ID": yt_id,
                "Video_URL": f"https://www.youtube.com/watch?v={yt_id}",
                "Video_Title": title,
                "Transcript_File": str(txt_path),
                "Needs_Paste": needs_paste,
                "Current_File_Preview_200c": current_preview,
            })
        except Exception:
            continue

    df = pd.DataFrame(rows)

    # Keep only those that still need paste
    if not df.empty:
        df = df[df["Needs_Paste"] == True].copy()
        df = df.sort_values(["Channel_Name", "DFIR_ID"])

    csv_path = out_path / "PASTE_TODO.csv"
    xlsx_path = out_path / "PASTE_TODO.xlsx"
    df.to_csv(csv_path, index=False, encoding="utf-8")
    try:
        df.to_excel(xlsx_path, index=False)
    except Exception:
        xlsx_path = None

    return df, csv_path, xlsx_path


todo_df, todo_csv, todo_xlsx = build_paste_todo(
    "/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts"
)

print("Saved:", todo_csv)
if todo_xlsx:
    print("Saved:", todo_xlsx)

todo_df.head(20)

Saved: /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts/PASTE_TODO.csv
Saved: /home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts/PASTE_TODO.xlsx


,Channel_ID,Channel_Name,DFIR_ID,YouTube_Video_ID,Video_URL,Video_Title,Transcript_File,Needs_Paste,Current_File_Preview_200c
161,C41,Belkasoft,V0363,Jsd2rGqVnfk,https://www.youtube.com/watch?v=Jsd2rGqVnfk,BelkaCTF #5 Solutions Video: PARTY GIRL—MISSING,/home/henrykabuye/Ransap/Publications 2026/Pro...,True,[NO TRANSCRIPT AVAILABLE]\nProvider: local_asr...
252,C33,ESET,V0293,xpk-sqi0eFA,https://www.youtube.com/watch?v=xpk-sqi0eFA,ESET Threat Report H2 2025 Webinar | ESET Rese...,/home/henrykabuye/Ransap/Publications 2026/Pro...,True,[NO TRANSCRIPT AVAILABLE]\nProvider: local_asr...
258,C33,ESET,V0298,Htfzf6W5ACM,https://www.youtube.com/watch?v=Htfzf6W5ACM,What's new in ESET PROTECT for business (Sprin...,/home/henrykabuye/Ransap/Publications 2026/Pro...,True,[NO TRANSCRIPT AVAILABLE]\nProvider: local_asr...
260,C33,ESET,V0302,4mn7GT7_mYs,https://www.youtube.com/watch?v=4mn7GT7_mYs,Science and Innovation: Tackling Climate and H...,/home/henrykabuye/Ransap/Publications 2026/Pro...,True,[NO TRANSCRIPT AVAILABLE]\nProvider: local_asr...
157,C17,FIRST Conference,V0160,UQ2TixVmGBE,https://www.youtube.com/watch?v=UQ2TixVmGBE,Automated ATT&CK Technique Chaining,/home/henrykabuye/Ransap/Publications 2026/Pro...,True,[NO TRANSCRIPT AVAILABLE]\nProvider: local_asr...
51,C49,LogRhythm,V0432,Diwn9whxgW8,https://www.youtube.com/watch?v=Diwn9whxgW8,Birth of a Whale,/home/henrykabuye/Ransap/Publications 2026/Pro...,True,[NO TRANSCRIPT AVAILABLE]\nProvider: local_asr...
92,C31,Mandiant,V0276,zSgK1J_Ijbs,https://www.youtube.com/watch?v=zSgK1J_Ijbs,"EP20 Windows Under the Hood: Kernel Design, ED...",/home/henrykabuye/Ransap/Publications 2026/Pro...,True,[NO TRANSCRIPT AVAILABLE]\nProvider: local_asr...
189,C06,Mandiant (Google Cloud Security),V0054,zSgK1J_Ijbs,https://www.youtube.com/watch?v=zSgK1J_Ijbs,"EP20 Windows Under the Hood: Kernel Design, ED...",/home/henrykabuye/Ransap/Publications 2026/Pro...,True,[NO TRANSCRIPT AVAILABLE]\nProvider: local_asr...
288,C15,RSA Conference,V0133,G_vIAf3dEUU,https://www.youtube.com/watch?v=G_vIAf3dEUU,RSAC Innovation Showcase: AI-Informed Policy G...,/home/henrykabuye/Ransap/Publications 2026/Pro...,True,[NO TRANSCRIPT AVAILABLE]\nProvider: local_asr...
283,C15,RSA Conference,V0134,BC8y69TpIbk,https://www.youtube.com/watch?v=BC8y69TpIbk,RSAC Innovation Showcase: Coding with AI: Can ...,/home/henrykabuye/Ransap/Publications 2026/Pro...,True,[NO TRANSCRIPT AVAILABLE]\nProvider: local_asr...


In [4]:
from pathlib import Path

def transcript_progress(out_dir):
    out_path = Path(out_dir)
    txts = list(out_path.rglob("V*.txt"))

    total = len(txts)
    done = 0

    for p in txts:
        try:
            t = p.read_text(encoding="utf-8", errors="ignore")
            if "NO TRANSCRIPT AVAILABLE" not in t and len(t.strip()) > 50:
                done += 1
        except:
            pass

    pct = (done / total * 100) if total else 0
    print(f"Completed: {done}/{total}  ({pct:.2f}%)")

transcript_progress("/home/henrykabuye/Ransap/Publications 2026/Proposal_2026/Knowledge_Agent/transcripts")

Completed: 421/440  (95.68%)
